In [ ]:
!cat /etc/os-release | head -2

PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"


In [ ]:
!whoami

root


In [ ]:
!ps aux | wc -l

19


In [ ]:
import torch, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
p = torch.cuda.get_device_properties(0)
print(f"{p.name} | compute capability sm_{p.major}{p.minor} | {p.total_memory/1e9:.2f} GB")
print("torch", torch.__version__)

Tesla T4, 15360 MiB, 580.82.07
Tesla T4 | compute capability sm_75 | 15.64 GB
torch 2.11.0+cu128


In [ ]:
MODEL = "WeiboAI/VibeThinker-3B"

In [ ]:
SHARED_PROMPT = "In one sentence, what is a data centre for?"

In [ ]:
YOUR_PROMPT = "You are a terse systems engineer. Answer in one sentence, no preamble, no bullet points. Question: What is a data centre for?"

In [ ]:
import time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained( MODEL, torch_dtype=torch.float16, device_map="cuda",trust_remote_code=True)                                # some models ship their own modelling code

def ask(prompt, max_new_tokens=1000):
    messages = [{"role": "user", "content": prompt + "\nAnswer with only the final answer in one sentence."}]
    inputs = tok.apply_chat_template(messages,add_generation_prompt=True,tokenize=True,return_dict=True,return_tensors="pt").to("cuda")
    t0 = time.time()
    out = model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=True,temperature=0.8)
    dt = time.time() - t0
    input_len = inputs["input_ids"].shape[-1]
    raw_answer = tok.decode(out[0][input_len:],skip_special_tokens=True).strip()
    if "</think>" in raw_answer:
        answer = raw_answer.split("</think>")[-1].strip()
    else:
        answer = raw_answer
    answer = " ".join(answer.splitlines())
    n = out.shape[-1] - input_len
    print(f"\n--- {prompt}\n{answer}\n({n} tokens in {dt:.1f}s, {n/dt:.1f} tok/s)")
    return answer

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [ ]:
print("model:", MODEL)
ask(YOUR_PROMPT)

model: WeiboAI/VibeThinker-3B

--- You are a terse systems engineer. Answer in one sentence, no preamble, no bullet points. Question: What is a data centre for?
A data centre is a dedicated facility that provides robust computing resources, storage, and networking capabilities for organizations.
(442 tokens in 28.5s, 15.5 tok/s)


'A data centre is a dedicated facility that provides robust computing resources, storage, and networking capabilities for organizations.'

In [ ]:
prompts = [
    "What is a data centre for?",
    "What is a GPU for?",
    "What is a container for?",
]

# one at a time
t0 = time.time()
for p in prompts:
    ask(p, max_new_tokens=60)
print(f"sequential: {time.time()-t0:.1f}s")

# all at once
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

texts = [
    tok.apply_chat_template(
        [{"role": "user", "content": p + "\nAnswer with only the final answer in one sentence."}],
        add_generation_prompt=True,
        tokenize=False
    )
    for p in prompts
]

t0 = time.time()
batch = tok(texts, return_tensors="pt", padding=True).to("cuda")
out = model.generate(**batch, max_new_tokens=60, do_sample=False)
print(f"batched: {time.time()-t0:.1f}s")

for row in out:
    print(tok.decode(row, skip_special_tokens=True)[:120])


--- What is a data centre for?
<think>The user asks: "What is a data centre for? Answer with only the final answer in one sentence."  We need to answer in one sentence, presumably concise. The question: "What is a data centre for?" We can answer: "A data centre is a facility that houses the hardware
(60 tokens in 3.1s, 19.5 tok/s)

--- What is a GPU for?
<think>The user asks: "What is a GPU for? Answer with only the final answer in one sentence."  We need to answer in exactly one sentence, and only the final answer. Probably "A GPU (graphics processing unit) is for processing graphics, rendering images, and performing computationally intensive tasks
(60 tokens in 3.1s, 19.6 tok/s)

--- What is a container for?
<think>We are asked: "What is a container for?" answer with only the final answer in one sentence. So we need to respond with a one-sentence answer that gives the answer. The question: "What is a container for?" Likely expecting: "A container for something or something else."
(6